## Treeview - Eine Tabelle erstellen

Tabellen gehören zu den wichtigsten Widgets einer Grafischen Oberfläche.

Beispiel einer einfachen Tabelle:

In [ ]:
import tkinter as tk
from tkinter import ttk

root = tk.Tk()
root.title("Tabelle mit Treeview")

# ---- Frame für Tabelle + Scrollbar ----
frame = ttk.Frame(root)
frame.pack(padx=10, pady=10)

# Scrollbar
scrollbar = ttk.Scrollbar(frame)
scrollbar.pack(side=tk.RIGHT, fill=tk.Y)

# Treeview (Tabelle)
columns = ("Name", "Alter", "Stadt")
tree = ttk.Treeview(frame, columns=columns, show="headings", yscrollcommand=scrollbar.set)

# Spaltenüberschriften
for col in columns:
    tree.heading(col, text=col)
    tree.column(col, width=120)

tree.pack()

scrollbar.config(command=tree.yview)

# ---- Tabellendaten einfügen ----
daten = [
    ("Anna", 29, "Berlin"),
    ("Bernd", 42, "Hamburg"),
    ("Clara", 31, "München"),
]

for row in daten:
    tree.insert("", tk.END, values=row)

root.mainloop()


#### Auswahl (Selection) in Treeview

Treeviews haben eine integrierte Auswahlverwaltung.

Werte der ausgewählten Zeile kann man so lesen:

In [ ]:
auswahl = tree.selection()
if auswahl:
    werte = tree.item(auswahl[0], "values")
    print(werte)

#### Ereignisse / Signale Events:

| Event                | Bedeutung                                       |
| -------------------- | ----------------------------------------------- |
| `<<TreeviewSelect>>` | Wird ausgelöst, wenn eine Zeile ausgewählt wird |
| `<Double-1>`         | Doppelklick auf eine Zelle                      |
| `<Button-1>`         | einfacher Mausklick                             |
| `<Return>`           | Enter-Taste                                     |
| `<FocusIn>`          | Fokus liegt nun im Treeview                     |


#### Auf Zeilenauswahl reagieren

In [ ]:
def on_select(event):
    auswahl = tree.selection()
    if auswahl:
        print("Ausgewählt:", tree.item(auswahl[0], "values"))

tree.bind("<<TreeviewSelect>>", on_select)


#### Reagiere auf Doppelklick

In [ ]:
def on_double_click(event):
    item_id = tree.identify_row(event.y)
    if item_id:
        print("Doppelklick auf:", tree.item(item_id, "values"))

tree.bind("<Double-1>", on_double_click)


#### Bestimmte Zelle identifizieren
Wenn du genau wissen willst, welche Zelle angeklickt wurde:

In [ ]:
row_id = tree.identify_row(event.y)
column_id = tree.identify_column(event.x)

print("Zeile:", row_id, "Spalte:", column_id)

werte = tree.item(row_id, "values")
print("Zellenwert:", werte[int(column_id[1:]) - 1])


#### Auswahlmöglichkeiten konfigurieren:

In [ ]:
tree.configure(selectmode="browse")  # Nur eine Zeile auswählbar

tree.configure(selectmode="extended")  # Mehrere Zeilen auswählbar

tree.configure(selectmode="none")  # Keine Auswahl erlaubt

In [ ]:
import tkinter as tk
from tkinter import ttk

root = tk.Tk()
root.title("Treeview Events")

columns = ("Name", "Alter", "Stadt")
tree = ttk.Treeview(root, columns=columns, show="headings", selectmode="browse")
tree.pack(fill="both", expand=True, padx=10, pady=10)

for col in columns:
    tree.heading(col, text=col)
    tree.column(col, width=120)

daten = [
    ("Anna", 29, "Berlin"),
    ("Bernd", 42, "Hamburg"),
    ("Clara", 31, "München"),
]

for row in daten:
    tree.insert("", tk.END, values=row)

# Auswahl-Ereignis
def on_select(event):
    item = tree.selection()
    if item:
        print("Ausgewählt:", tree.item(item[0], "values"))

tree.bind("<<TreeviewSelect>>", on_select)

# Doppelklick
def on_double(event):
    row = tree.identify_row(event.y)
    col = tree.identify_column(event.x)
    print("Doppelklick auf Zelle:", row, col)

tree.bind("<Double-1>", on_double)

root.mainloop()


#### Editierbare Zellen
Eine Zelle kann durch Doppelklick geändert werden. Bei Verlust des Focus oder Drücken auf "Return" wird die Änderung gespeichert

In [2]:
import sqlite3
import tkinter as tk
from tkinter import ttk

DB = "telefonbuch.db"  # Pfad zu deiner SQLite-Datei
TABLE = "telefonbuch"

# -----------------------------
# Datenbank-Funktionen
# -----------------------------
def get_all_contacts():
    con = sqlite3.connect(DB)
    cur = con.cursor()
    cur.execute(f"SELECT id, vorname, nachname, vorwahl, rufnummer FROM {TABLE}")
    rows = cur.fetchall()
    con.close()
    return rows

def update_contact(id_, column_name, new_value):
    con = sqlite3.connect(DB)
    cur = con.cursor()
    cur.execute(f"UPDATE {TABLE} SET {column_name} = ? WHERE id = ?", (new_value, id_))
    con.commit()
    con.close()

# -----------------------------
# Editierbare Zelle
# -----------------------------
def edit_cell(event):
    row_id = tree.identify_row(event.y)
    col_id = tree.identify_column(event.x)

    if not row_id or col_id == "#0":
        return

    col_index = int(col_id[1:]) - 1
    column_name = columns[col_index]

    # DB-ID der Zeile
    item = tree.item(row_id)
    db_id = item["values"][0]

    x, y, w, h = tree.bbox(row_id, col_id)
    aktueller_wert = item["values"][col_index]

    entry = tk.Entry(root)
    entry.place(x=x + tree.winfo_rootx() - root.winfo_rootx(),
                y=y + tree.winfo_rooty() - root.winfo_rooty(),
                width=w, height=h)
    entry.insert(0, aktueller_wert)
    entry.focus()

    def save_edit(event=None):
        neuer_wert = entry.get()
        entry.destroy()
        values = list(tree.item(row_id, "values"))
        values[col_index] = neuer_wert
        tree.item(row_id, values=values)
        update_contact(db_id, column_name, neuer_wert)

    entry.bind("<Return>", save_edit)
    entry.bind("<FocusOut>", save_edit)

# -----------------------------
# GUI
# -----------------------------
root = tk.Tk()
root.title("Telefonbuch")

columns = ("id", "vorname", "nachname", "vorwahl", "rufnummer")

tree = ttk.Treeview(root, columns=columns, show="headings")
tree.pack(fill="both", expand=True, padx=10, pady=10)

for col in columns:
    tree.heading(col, text=col.capitalize())
    tree.column(col, width=120)

# Daten laden
for row in get_all_contacts():
    tree.insert("", tk.END, values=row)

# Doppelklick bind
tree.bind("<Double-1>", edit_cell)

root.mainloop()


Wir nutzen:

* <code>identify_row(event.y)</code>

* <code>identify_column(event.x)</code>

* <code>bbox()</code> um die genaue Position der Zelle zu bekommen

* ein **Entry**, das über der Zelle sitzt

#### Gleiches Beispiel OOP

In [1]:
import sqlite3
import tkinter as tk
from tkinter import ttk
from pathlib import Path

# Konfiguration
ordner = Path(__file__).parent
DB_NAME = ordner / "telefonbuch.db"
TABLE_NAME = "telefonbuch"

class TelefonbuchDB:
    """Klasse für alle Datenbankoperationen"""
    def __init__(self, db_name, table_name):
        self.db_name = db_name
        self.table_name = table_name

    def get_connection(self):
        """Erstellt eine Datenbankverbindung"""
        return sqlite3.connect(self.db_name)

    def get_all_entries(self):
        """Holt alle Einträge aus der Datenbank"""
        con = self.get_connection()
        cur = con.cursor()
        try:
            cur.execute(f"SELECT id, vorname, nachname, vorwahl || ' ' || rufnummer as telefon FROM {self.table_name}")
            rows = cur.fetchall()
            return rows
        except sqlite3.Error as e:
            print(f"Datenbankfehler: {e}")
            return []
        finally:
            con.close()

    def update_entry(self, entry_id, column_name, new_value):
        """Aktualisiert einen Eintrag in der Datenbank"""
        con = self.get_connection()
        cur = con.cursor()
        query = f"UPDATE {self.table_name} SET {column_name} = ? WHERE id = ?"
        cur.execute(query, (new_value, entry_id))
        con.commit()
        con.close()
        print(f"Update: ID {entry_id}, {column_name} -> {new_value}")

class TelefonbuchUI:
    """Klasse für die Benutzeroberfläche"""
    def __init__(self, root, db):
        """
        Der Konstruktor: Wird einmal aufgerufen, wenn die App startet.
        Hier bauen wir das Fenster und laden die Daten.
        """
        self.root = root
        self.db = db
        self.root.title("Telefonbuch")

        # Spalten definieren (angepasst an die DB-Struktur: 'telefon' statt 'vorwahl', 'rufnummer')
        self.columns = ("id", "vorname", "nachname", "telefon")

        # 1. Benutzeroberfläche (GUI) aufbauen
        self.setup_gui()

        # 2. Daten aus der Datenbank laden
        self.load_data()

    def setup_gui(self):
        """Erstellt die Tabelle (Treeview) und Scrollbars."""
        self.tree = ttk.Treeview(self.root, columns=self.columns, show="headings")
        self.tree.pack(fill="both", expand=True, padx=10, pady=10)

        # Spaltenüberschriften setzen
        for col in self.columns:
            # anchor="w" sorgt dafür, dass die Überschrift linksbündig ist
            self.tree.heading(col, text=col.capitalize(), anchor="w")
            self.tree.column(col, width=100) # Standardbreite

        # Individuelle Breitenanpassung (einfacher für Teilnehmer)
        self.tree.column("id", width=50)
        self.tree.column("vorname", width=150)
        self.tree.column("nachname", width=150)
        self.tree.column("telefon", width=120)

        # Doppelklick-Event verknüpfen
        self.tree.bind("<Double-1>", self.on_double_click)

    def load_data(self):
        """Holt Daten aus der DB und füllt die Tabelle."""
        # Tabelle leeren, falls man neu lädt
        for item in self.tree.get_children():
            self.tree.delete(item)

        # Daten von der Datenbank holen
        rows = self.db.get_all_entries()
        for row in rows:
            self.tree.insert("", tk.END, values=row)

    def on_double_click(self, event):
        """Startet den Editiermodus für eine Zelle."""
        # Wo wurde geklickt?
        row_id = self.tree.identify_row(event.y)
        col_id = self.tree.identify_column(event.x)

        if not row_id or col_id == "#0":
            return

        # Spalten-Index und Name ermitteln
        # col_id ist z.B. "#1", wir brauchen int 0
        col_index = int(col_id[1:]) - 1
        column_name = self.columns[col_index]

        # Werte der Zeile holen
        item = self.tree.item(row_id)
        row_values = item["values"]
        db_id = row_values[0] # ID ist immer an erster Stelle
        current_value = row_values[col_index]

        # Position für das Eingabefeld berechnen
        x, y, w, h = self.tree.bbox(row_id, col_id)

        # Entry-Widget erstellen
        entry = tk.Entry(self.root)
        # Position relativ zum Root-Fenster korrigieren
        entry.place(x=x + self.tree.winfo_rootx() - self.root.winfo_rootx(),
                    y=y + self.tree.winfo_rooty() - self.root.winfo_rooty(),
                    width=w, height=h)

        entry.insert(0, current_value)
        entry.focus()

        def save_edit(event=None):
            new_value = entry.get()
            entry.destroy()

            # 1. Update in der GUI (Treeview)
            # Wir müssen die Liste der Werte kopieren und bearbeiten
            new_values = list(self.tree.item(row_id, "values"))
            new_values[col_index] = new_value
            self.tree.item(row_id, values=new_values)

            # 2. Update in der Datenbank
            self.db.update_entry(db_id, column_name, new_value)

        # Speichern bei Enter oder wenn man wegklickt
        entry.bind("<Return>", save_edit)
        entry.bind("<FocusOut>", save_edit)

# -----------------------------
# Hauptprogramm
# -----------------------------
if __name__ == "__main__":
    # 1. Datenbankinstanz erstellen
    db = TelefonbuchDB(DB_NAME, TABLE_NAME)

    # 2. Tkinter-Fenster erstellen
    root = tk.Tk()

    # 3. UI-Instanz erstellen und Datenbank übergeben
    app = TelefonbuchUI(root, db)

    # 4. Hauptschleife starten
    root.mainloop()

Update: ID 7, nachname -> f


<img style="float: center;" src="img/wbs-logo.jpg">


Autor: Dirk Maric
### Abbildungs- und Quellenverzeichnis
https://de.wikipedia.org/wiki/Python_(Programmiersprache)
Das Python Logo ist ein eingetragenes Warenzeichen der Python Software Foundation
Alle auf dieser Website veröffentlichten Logos sowie Marken-, Produkt- und Warenzeichen sind Eigentum der jeweiligen Unternehmen
© WBS TRAINING AG – Alle Rechte vorbehalten

### Nutzungsrechte:
Die Nutzung dieser Dokumentation ist ausschließlich für Schulungszwecke der WBS TRAINING AG gestattet. Eine Weitergabe an Dritte, auch auszugsweise, sowie Vervielfältigungen und Verbreitungen aller Art (elektronische und andere Verfahren) inklusive Übersetzungen sind nur mit vorheriger schriftlicher Zustimmung des Rechtinhabers gestattet. Zuwiderhandlungen verpflichten zu Schadenersatz.

### Herausgeber:

WBS TRAINING AG
Lorenzweg 5
12099 Berlin
Haftungsausschluss:
Alle Inhalte sind nach bestem Wissen korrekt und vollständig recherchiert und mit größtmöglicher Sorgfalt für die Schulungsunterlage zusammengestellt. Wir sind um die laufende Aktualisierung aller Informationen und Daten bemüht. Dennoch können Fehler (z.B. Abweichungen zur beschriebenen Hard- und Software durch kurzfristige Updates) auftreten, sodass wir für die vollständige Übereinstimmung, Richtigkeit und Aktualität keine Gewähr übernehmen. Hinweise unserer Nutzer werden konsequent weiterverfolgt.
